## Data ingestion

For data ingestion, a typical workflow is the following:

RAW SOURCE

Steel_industry_data.csv

        ↓
        
02 — INGESTION + DATA QUALITY

        ↓
        
RAW DATABASE TABLE

raw_energy_usage

        ↓
        
03 — SQL TRANSFORMATION

        ↓
        
CLEAN / CURATED TABLE

clean_energy_usage

        ↓
        
OPTIONAL ANALYTICAL / FEATURE TABLE

energy_model_features

        ↓
        
pandas

        ↓
        
EDA

        ↓
        
ML preprocessing

        ↓
        
models

        ↓
        
validation / optimization


02 — Data Ingestion & Quality

1. Connect to DuckDB
2. Load raw CSV into raw table

3. Schema validation
   DESCRIBE

4. Row-count validation
   COUNT(*)

5. Missing-value validation
   COUNT(column)

6. Duplicate validation
   exact duplicates
   duplicate timestamps

7. Timestamp validation
   TRY_STRPTIME

8. Category validation
   GROUP BY
   DISTINCT

9. Numerical/domain validation
   MIN/MAX
   WHERE conditions

10. Temporal coverage validation
    MIN/MAX timestamp
    distinct days
    observations/day
    LAG()

11. Cross-column consistency
    Day_of_week vs timestamp
    WeekStatus vs timestamp

12. Data-quality conclusions

In [3]:
#1. Connect do DuckDB

#Creating a local database first

import duckdb

con = duckdb.connect("Steel_energy.duckdb")

#2. Creating the raw data table in Steel_energy.duckdb database

con.sql("""
    CREATE OR REPLACE TABLE raw_energy_usage AS
    SELECT *
    FROM read_csv_auto('Steel_industry_data.csv')
""")

In [9]:
#3. Schema validation with describe
con.sql("""
    DESCRIBE raw_energy_usage;
""")

┌──────────────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│             column_name              │ column_type │  null   │   key   │ default │  extra  │
│               varchar                │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date                                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ Usage_kWh                            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Lagging_Current_Reactive.Power_kVarh │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Leading_Current_Reactive_Power_kVarh │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ CO2(tCO2)                            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Lagging_Current_Power_Factor         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Leading_Current_Power_Factor         │ DOUBLE   

#It looks like date is a VARCHAR, not a date format, so we will likely need to convert it later during transformation

In [10]:
#4. Row count validation
con.sql("""
    SELECT COUNT(*)
    FROM raw_energy_usage
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        35040 │
└──────────────┘

In [12]:
#5. Missing values
#You could check for 1 column or for multiple columns as well

con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(Usage_kWh) AS usage_non_null
    FROM raw_energy_usage
""")

┌───────┬────────────────┐
│ rows  │ usage_non_null │
│ int64 │     int64      │
├───────┼────────────────┤
│ 35040 │          35040 │
└───────┴────────────────┘

In [13]:
#No null rows for the target, good

In [16]:
#Checking all the other columns
con.sql("""
SELECT
    COUNT(*) - COUNT(Usage_kWh)
        AS usage_nulls,
    COUNT(*) - COUNT("Lagging_Current_Reactive.Power_kVarh")
        AS lagging_reactive_nulls
    FROM raw_energy_usage
""")

┌─────────────┬────────────────────────┐
│ usage_nulls │ lagging_reactive_nulls │
│    int64    │         int64          │
├─────────────┼────────────────────────┤
│           0 │                      0 │
└─────────────┴────────────────────────┘

In [17]:
#Can be extended to all the columns

In [18]:
#6. Duplicate validation
# Identical rows
con.sql("""
    SELECT
        *,
        COUNT(*) AS duplicate_count
    FROM raw_energy_usage
    GROUP BY ALL
    HAVING COUNT(*) > 1
""")

┌─────────┬───────────┬──────────────────────────────────────┬──────────────────────────────────────┬───────────┬──────────────────────────────┬──────────────────────────────┬───────┬────────────┬─────────────┬───────────┬─────────────────┐
│  date   │ Usage_kWh │ Lagging_Current_Reactive.Power_kVarh │ Leading_Current_Reactive_Power_kVarh │ CO2(tCO2) │ Lagging_Current_Power_Factor │ Leading_Current_Power_Factor │  NSM  │ WeekStatus │ Day_of_week │ Load_Type │ duplicate_count │
│ varchar │  double   │                double                │                double                │  double   │            double            │            double            │ int64 │  varchar   │   varchar   │  varchar  │      int64      │
└─────────┴───────────┴──────────────────────────────────────┴──────────────────────────────────────┴───────────┴──────────────────────────────┴──────────────────────────────┴───────┴────────────┴─────────────┴───────────┴─────────────────┘
                                    

#Sometimes different rows might have the same exact timestamp. You need to investigate if that happens. It might be a duplicate row to be removed, or observations from different sensors/machines

In [21]:
con.sql("""
    SELECT 
        date,
        COUNT(*) AS observations
    FROM raw_energy_usage
    GROUP BY date
    HAVING COUNT(*) > 1
    ORDER BY observations DESC;
""")

┌─────────┬──────────────┐
│  date   │ observations │
│ varchar │    int64     │
└─────────┴──────────────┘
          0 rows        

In [22]:
#No duplicate timestamps

#7. Timestamp validation
#If the timestamp data type is not in the proper format, you need to standardize it later when running transformations

In [25]:
con.sql("""
    SELECT date
    FROM raw_energy_usage
    LIMIT 10
""")

┌──────────────────┐
│       date       │
│     varchar      │
├──────────────────┤
│ 01/01/2018 00:15 │
│ 01/01/2018 00:30 │
│ 01/01/2018 00:45 │
│ 01/01/2018 01:00 │
│ 01/01/2018 01:15 │
│ 01/01/2018 01:30 │
│ 01/01/2018 01:45 │
│ 01/01/2018 02:00 │
│ 01/01/2018 02:15 │
│ 01/01/2018 02:30 │
└──────────────────┘
      10 rows     

In [26]:
#from a first inspection, it is a varchar. 15 min intervals

In [29]:
#Let's try to conver it to a proper timestamp
con.sql("""
    SELECT
        date,
        STRPTIME(date, '%d/%m/%Y %H:%M') AS timestamp
    FROM raw_energy_usage
    LIMIT 10;
""")

┌──────────────────┬─────────────────────┐
│       date       │      timestamp      │
│     varchar      │      timestamp      │
├──────────────────┼─────────────────────┤
│ 01/01/2018 00:15 │ 2018-01-01 00:15:00 │
│ 01/01/2018 00:30 │ 2018-01-01 00:30:00 │
│ 01/01/2018 00:45 │ 2018-01-01 00:45:00 │
│ 01/01/2018 01:00 │ 2018-01-01 01:00:00 │
│ 01/01/2018 01:15 │ 2018-01-01 01:15:00 │
│ 01/01/2018 01:30 │ 2018-01-01 01:30:00 │
│ 01/01/2018 01:45 │ 2018-01-01 01:45:00 │
│ 01/01/2018 02:00 │ 2018-01-01 02:00:00 │
│ 01/01/2018 02:15 │ 2018-01-01 02:15:00 │
│ 01/01/2018 02:30 │ 2018-01-01 02:30:00 │
└──────────────────┴─────────────────────┘
  10 rows                      2 columns

In [30]:
#Conversion was successful, we'll need that during transformations

In [31]:
#If conversion crashes, you need TRY_STRPTIME

In [35]:
con.sql("""
    SELECT *
    FROM raw_energy_usage
    WHERE TRY_STRPTIME(
        date,
        '%d/%m/%Y %H:%M'
        ) IS NULL;
""")

┌─────────┬───────────┬──────────────────────────────────────┬──────────────────────────────────────┬───────────┬──────────────────────────────┬──────────────────────────────┬───────┬────────────┬─────────────┬───────────┐
│  date   │ Usage_kWh │ Lagging_Current_Reactive.Power_kVarh │ Leading_Current_Reactive_Power_kVarh │ CO2(tCO2) │ Lagging_Current_Power_Factor │ Leading_Current_Power_Factor │  NSM  │ WeekStatus │ Day_of_week │ Load_Type │
│ varchar │  double   │                double                │                double                │  double   │            double            │            double            │ int64 │  varchar   │   varchar   │  varchar  │
└─────────┴───────────┴──────────────────────────────────────┴──────────────────────────────────────┴───────────┴──────────────────────────────┴──────────────────────────────┴───────┴────────────┴─────────────┴───────────┘
                                                                                                            

In [36]:
#0 rows, conversion works on every date

#if dates are malformed, it is an upstream data-quality problem, which you need to fix at the source

In [38]:
#8. Category validation
#For each categorical column, you want to inspect categories
#You can check what categories you have, and the presence of errors

con.sql("""
    SELECT
        Load_Type,
        COUNT(*) AS n
    FROM raw_energy_usage
    GROUP BY Load_Type;
""")

┌──────────────┬───────┐
│  Load_Type   │   n   │
│   varchar    │ int64 │
├──────────────┼───────┤
│ Medium_Load  │  9696 │
│ Maximum_Load │  7272 │
│ Light_Load   │ 18072 │
└──────────────┴───────┘

#Looks good. If there are categories with errors or typos, you may need to do some standardization, for example with CASE/WHEN

In [40]:
#9. Numerical validation
#Some variables have constraints, in between MIN and MAX values
#For example, here energy usage can't be negative
#You should run some checks

con.sql("""
    SELECT *
    FROM raw_energy_usage
    WHERE Lagging_Current_Power_Factor < 0 
    OR Lagging_Current_Power_Factor > 100
""")

┌─────────┬───────────┬──────────────────────────────────────┬──────────────────────────────────────┬───────────┬──────────────────────────────┬──────────────────────────────┬───────┬────────────┬─────────────┬───────────┐
│  date   │ Usage_kWh │ Lagging_Current_Reactive.Power_kVarh │ Leading_Current_Reactive_Power_kVarh │ CO2(tCO2) │ Lagging_Current_Power_Factor │ Leading_Current_Power_Factor │  NSM  │ WeekStatus │ Day_of_week │ Load_Type │
│ varchar │  double   │                double                │                double                │  double   │            double            │            double            │ int64 │  varchar   │   varchar   │  varchar  │
└─────────┴───────────┴──────────────────────────────────────┴──────────────────────────────────────┴───────────┴──────────────────────────────┴──────────────────────────────┴───────┴────────────┴─────────────┴───────────┘
                                                                                                            

In [47]:
#10. Temporal coverage validation MIN/MAX timestamp distinct days observations/day LAG()
#After converting timestamps, check the time coverage
con.sql("""
    SELECT
        MIN(
            STRPTIME(date, '%d/%m/%Y %H:%M')
        ) AS first_timestamp,

        MAX(
            STRPTIME(date, '%d/%m/%Y %H:%M')
        ) AS last_timestamp
    FROM raw_energy_usage;
""")

┌─────────────────────┬─────────────────────┐
│   first_timestamp   │   last_timestamp    │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2018-01-01 00:00:00 │ 2018-12-31 23:45:00 │
└─────────────────────┴─────────────────────┘

In [42]:
#coverage is for the full 2018 year

In [45]:
#you can also inspect the number of observations per day
#time gaps
#if some time gaps are larger than expected, inspect

con.sql("""
    WITH ordered_data AS(
        SELECT 
            STRPTIME(
                date,
                '%d/%m/%Y %H:%M'
            )AS timestamp
        FROM raw_energy_usage
    ),

    time_gaps AS (
        SELECT
            timestamp,
            LAG(timestamp) OVER(
                ORDER BY timestamp
            ) AS previous_timestamp
        FROM ordered_data
    )

    SELECT 
        timestamp,
        previous_timestamp,
        timestamp - previous_timestamp AS time_gap
    FROM time_gaps
    WHERE timestamp - previous_timestamp
        <> INTERVAL '15 minutes'
""")

┌───────────┬────────────────────┬──────────┐
│ timestamp │ previous_timestamp │ time_gap │
│ timestamp │     timestamp      │ interval │
└───────────┴────────────────────┴──────────┘
                   0 rows                  

In [46]:
#All the time gaps are exactly of 15 minutes
#If different, check problems and take notes for transformation

In [48]:
#12. Conclusions
#Great data quality. Only timestamp conversion needed 

In [49]:
con.close()